In [26]:
from data_extraction.utils.rate_limit_handling import request_with_backoff, RateLimiter
musicbrainz_limiter = RateLimiter(calls_per_second=0.5)
def call_musicbrainz_search(artist_name: str):
    MUSICBRAINZ_ENDPOINT=f"https://musicbrainz.org/ws/2/artist/"
    headers = {"User-Agent": "ArabMusicMap/1.0 (yussef0212@gmail.com)"}
    params = {
        "query": f"artist:{artist_name}",
        "fmt": "json",
        "limit": 5
    }
    musicbrainz_limiter.wait()
    response = request_with_backoff(MUSICBRAINZ_ENDPOINT, params=params, headers=headers)
    if response.get('artists'):
        return response['artists']

In [25]:
# response = call_musicbrainz_search("wegz")
# import json
# print(json.dump(response))
from data_extraction.db_operations.get_features import execute, getcon
results = execute("select name, id from artists where mbid is null")
results = [result for result in results if result[1] not in (878, 759)]
print(len(results))

190


In [9]:
def name_to_mbid(id: int, mbid: str):
    con = getcon()
    return con.execute(
        """
        UPDATE artists
        SET 
            mbid = ?,
        WHERE id = ?;
        """
    ,(mbid, id)) 

In [19]:
resp = call_musicbrainz_search('Asala')
resp

{'created': '2026-08-08T09:08:11.538Z', 'count': 0, 'offset': 0, 'artists': []}

In [28]:
import pickle
pickle.dump(results, open('no_mbid_artists.pkl', 'wb'))


In [37]:
print(len(results))

190


In [43]:
def display(artists):
    st = ""
    for artist in artists:
        st += artist.get("name") + "\n"
        st += artist.get('id') + "\n"
        st += str(artist.get('score')) + "\n"  
        st += "\n"
    return st

In [ ]:
chek = ""
import tqdm
for item in tqdm.tqdm(results):
    artists = call_musicbrainz_search(item[0])
    artist = artists[0] if artists else None
    if artist and artist.get('id'):
        name_to_mbid(item[1], artist['id'])
        chek += f"ID: {item[1]} Artist: {item[0]}\n"
        chek += f"Artist URL: https://musicbrainz.org/artist/{artist['id']}\n"
        chek = chek + display(artists)
        continue
    print(f"Artist {item[0]} with id {item[1]} not found in musicbrainz")
    chek += f"Artist {item[0]} with id {item[1]} not found in musicbrainz"

with open("artists.txt", "w") as f:
    f.write(chek)

In [24]:
# results
for item in results:
    execute(f"update artists set mbid = null where id={item[1]}")

In [5]:
from data_extraction.db_operations.get_features import execute, getcon
getcon()
execute(
"""
delete from artist_similarity_lastfm where artist_id=1879;
delete from failed_artists where artist_id=1879;
delete from failed_tracks where artist_id=1879;
delete from tracks where artist_id=1879;
delete from artists where id = 1879;
""")

python-dotenv could not parse statement starting at line 8


IOException: IO Error: Could not set lock on file "/home/youssef/Documents/python/music-map/backend/data_extraction/music.duckdb": Conflicting lock is held in /usr/bin/python3.14 (PID 34371) by user youssef. See also https://duckdb.org/docs/stable/connect/concurrency

In [41]:
execute("select * from artists where id=1637 or id=1083")

[(1637,
  'dd8502a9-ce21-4a77-a0c8-fae3029c28cb',
  '458940829',
  'محمد الموجي',
  'محمد الموجي',
  'EG',
  23,
  [],
  datetime.datetime(2026, 7, 5, 18, 14, 7, 218101),
  'يحيى الموجي'),
 (1083,
  'ee79ff79-2ecc-449e-b982-367a0abd111b',
  '650294904',
  'اميمة الخليل',
  'اميمة الخليل',
  'MA',
  23,
  [],
  datetime.datetime(2026, 7, 5, 16, 29, 23, 486970),
  'Al Harraq, Khalil')]

In [40]:
mbid='dd8502a9-ce21-4a77-a0c8-fae3029c28cb'
id=1637
execute(f"update artists set mbid='{mbid}' where id = {id}")

[(1,)]

In [45]:
id = 2040
listeners = 3500
execute(f"update artists set mbid=null, lastfm_listeners={listeners} where id = {id}")

[(1,)]

In [ ]:
import requests
from data_extraction.utils.rate_limit_handling import request_with_backoff, RateLimiter
lastfm_limiter = RateLimiter(calls_per_second=3)
import os

def call_lastfm(artist_name: str, get_similar_artists: bool):

    LASTFM_API_ENDPOINT = "https://ws.audioscrobbler.com/2.0/"
    LAST_FM_API_KEY = os.getenv("LAST_FM_API_KEY")
    endpoint = "artist.search" if get_similar_artists else "artist.getinfo"

    headers = {"User-Agent": "ArabMusicMap/1.0 (yussef0212@gmail.com)"}
    params = {
        "method": endpoint,
        "artist": artist_name,
        "api_key": LAST_FM_API_KEY,
        "format": "json",
        "limit": 5
    }

    lastfm_limiter.wait()
    response = request_with_backoff(LASTFM_API_ENDPOINT, params=params, headers=headers)
    artists = response['results']['artistmatches']['artist']
    return artists[0]['listeners']

In [14]:
with open('no_mbid_artists.pkl', 'rb') as f:
    no_mbid_artists = pickle.load(f)
from data_extraction.db_operations.get_features import execute
from tqdm import tqdm
cnt = 0
for artist in tqdm(no_mbid_artists[145:]):
    try:
        listeners = call_lastfm(artist[0], True)
    except KeyError as e:
        listeners = None
        print(f"Issue with artist {artist[0]}, id {artist[1]}: {e}")
    except IndexError as e:
        print(f"Issue with artist {artist[0]}, id {artist[1]}: {e}")
    if listeners:
        execute(f"update artists set mbid=null, lastfm_listeners = {listeners} where id={artist[1]}")
        cnt += 1
print(f"{cnt} artists updated.")


  7%|▋         | 3/45 [00:01<00:17,  2.38it/s]

Issue with artist سالم طربيه, id 2126: list index out of range


 22%|██▏       | 10/45 [00:04<00:15,  2.30it/s]

Issue with artist عمر كيف, id 2285: list index out of range


100%|██████████| 45/45 [00:21<00:00,  2.11it/s]

45 artists updated.
